# VOLTA — Track D: Window creation | إنشاء الـ windows

**Goal:** create as many training windows as possible from little data, while keeping train and validation separate.

- Input window: **168 hours**. Target: the **next 24 hours**. A forecast is issued at **00:00 UTC**.
- Split by position (UTC): train 17,545 rows (to 2016-12-31 23:00), validation 8,760 rows (2017). Test (2018) is **not used**.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path

CANDIDATES = ["../data/processed/volta_clean_hourly.csv",
              "data/processed/volta_clean_hourly.csv",
              "volta_clean_hourly.csv"]
path = next(p for p in CANDIDATES if Path(p).exists())

df = pd.read_csv(path, index_col=0, parse_dates=True)
price = df["price actual"]
idx = price.index

W, H = 168, 24            # input window, forecast horizon (hours)
n_tr, n_va = 17545, 8760  # train 2015-2016, validation 2017

print(path, "|", len(price), "rows |", idx[0], "->", idx[-1])

../data/processed/volta_clean_hourly.csv | 35064 rows | 2014-12-31 23:00:00+00:00 -> 2018-12-31 22:00:00+00:00


### Window builder
`i` is the forecast issue time. Input = `[i-W, i)`, target = `[i, i+H)`. The **target** must stay inside its split; the **input** may reach back into the previous split (known at forecast time).

In [2]:
def window_positions(a, b, midnight_only=False):
    pos = []
    for i in range(max(a, W), b - H + 1):
        if midnight_only and idx[i].hour != 0:
            continue
        pos.append(i)
    return np.array(pos)

train_daily  = window_positions(0, n_tr, midnight_only=True)   # one window per day
train_hourly = window_positions(0, n_tr)                       # one window per hour (slide by 1 h)
val_daily    = window_positions(n_tr, n_tr + n_va, midnight_only=True)

print("train daily :", len(train_daily))
print("train hourly:", len(train_hourly))
print("val daily   :", len(val_daily))

train daily : 724
train hourly: 17354
val daily   : 365


### Checks: targets stay inside their split

In [3]:
assert (train_hourly + H <= n_tr).all()
assert (val_daily + H <= n_tr + n_va).all()

i = val_daily[0]
print("first validation window")
print("input :", idx[i-W], "->", idx[i-1])
print("target:", idx[i],   "->", idx[i+H-1])

first validation window
input : 2016-12-25 00:00:00+00:00 -> 2016-12-31 23:00:00+00:00
target: 2017-01-01 00:00:00+00:00 -> 2017-01-01 23:00:00+00:00


## Conclusions | الخلاصة

- Training windows: **724** (one per day) or **17,354** (sliding by 1 hour). Validation: **365** windows (one per day, same as the baselines).
- Every target stays inside its own split (checked with `assert`).
- The first validation window uses the last days of 2016 as input. This is allowed because those values were already known at forecast time.
- Next step: Track B trains the LSTM with both versions and compares them on validation.